# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thaianh20021/K4-Track-4-Day10-VisionLanguageModel/blob/main/lab_day10.ipynb)

Đọc `README.md` và `GUIDE.md` trước. Notebook này cài đặt hoàn chỉnh pipeline theo từng chặng của bài lab.

## 0. Môi trường (Colab T4)

In [ ]:
# 0. Kiểm tra thông tin GPU (khuyến nghị GPU T4 16 GB trên Colab)
!nvidia-smi -L

# 1. Đồng bộ repository và chuyển thư mục làm việc nếu chạy trên Google Colab
import os
if os.path.exists('/content'):
    if not os.path.exists('/content/K4-Track-4-Day10-VisionLanguageModel'):
        !git clone https://github.com/thaianh20021/K4-Track-4-Day10-VisionLanguageModel.git
    %cd /content/K4-Track-4-Day10-VisionLanguageModel

# 2. Mount Google Drive để lưu checkpoint adapter LoRA và projector
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# 3. Cài đặt các thư viện phụ thuộc theo phiên bản đã thẩm định tương thích
!pip -q install -r requirements-colab.txt

# 4. Kiểm tra tính toàn vẹn của công cụ đề bài (kỳ vọng: 68 passed, 1 skipped)
!python -m pytest -q

In [ ]:
# Cấu hình Kaggle API credentials từ Colab Secrets (KHÔNG ghi khóa vào notebook)
import os

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print(f"Đã nạp Kaggle username: {os.environ.get('KAGGLE_USERNAME')}")
except Exception as e:
    print(f"Lưu ý: Chưa lấy được secret Kaggle từ Colab Secrets ({e}).")
    print("Cách 1: Thêm secret KAGGLE_USERNAME và KAGGLE_KEY vào biểu tượng 🔑 Secrets ở thanh bên trái của Colab.")
    print("Cách 2: Nếu chưa dùng Secrets, bạn có thể gán tạm thời: os.environ['KAGGLE_USERNAME'] = '...'; os.environ['KAGGLE_KEY'] = '...'")

## 1. Dữ liệu: tải và chia split

In [ ]:
# Chạy prepare_data.py để tải dataset và chia split cố định theo phiên quay (session)
# Nếu tải zip thủ công: !python scripts/prepare_data.py --zip <đường_dẫn_zip>
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scripts.labcommon import CLASSES, read_jsonl

# 1. Đọc và kiểm tra metadata của split
meta_path = Path("data/splits/meta.json")
assert meta_path.exists(), "Không tìm thấy data/splits/meta.json! Hãy chạy prepare_data.py trước."
meta = json.loads(meta_path.read_text(encoding="utf-8"))
dataset_version = meta.get("dataset_version", "")
print("=" * 65)
print(f"Dataset Version: {dataset_version}")
assert dataset_version.startswith("d10-"), f"dataset_version không bắt đầu bằng 'd10-': {dataset_version}"
print(f"Chữ ký dataset_version hợp lệ: {dataset_version}")
print("=" * 65)

# 2. Đọc trực tiếp các split .jsonl và đối chiếu số lượng ảnh và box từng lớp
splits_data = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
expected = {
    "train": {"images": 1422, "pothole": 1074, "crack": 1750, "manhole": 691},
    "val":   {"images": 276,  "pothole": 66,   "crack": 470,  "manhole": 103},
    "test":  {"images": 311,  "pothole": 121,  "crack": 299,  "manhole": 163},
}

print(f"{'Split':<7} | {'Ảnh':<6} | {'Pothole':<8} | {'Crack':<8} | {'Manhole':<8} | {'Trạng thái'}")
print("-" * 65)
for s in ("train", "val", "test"):
    rows = splits_data[s]
    n_imgs = len(rows)
    box_counts = {c: 0 for c in CLASSES}
    for r in rows:
        for obj in r["objects"]:
            box_counts[obj["label"]] += 1
    
    exp = expected[s]
    is_ok = (n_imgs == exp["images"] and all(box_counts[c] == exp[c] for c in CLASSES))
    status_str = "CHÍNH XÁC (PASS)" if is_ok else "LỆCH (FAIL)"
    print(f"{s:<7} | {n_imgs:<6} | {box_counts['pothole']:<8} | {box_counts['crack']:<8} | {box_counts['manhole']:<8} | {status_str}")
    
    assert n_imgs == exp["images"], f"Lệch số ảnh {s}: {n_imgs} vs {exp['images']}"
    for c in CLASSES:
        assert box_counts[c] == exp[c], f"Lệch số box {c} trong {s}: {box_counts[c]} vs {exp[c]}"

# 3. Kiểm tra tính độc lập giữa các phiên quay (Chống rò rỉ - Data Leakage)
train_sessions = {r["session"] for r in splits_data["train"]}
val_sessions = {r["session"] for r in splits_data["val"]}
test_sessions = {r["session"] for r in splits_data["test"]}

overlap_train_val = train_sessions & val_sessions
overlap_train_test = train_sessions & test_sessions
overlap_val_test = val_sessions & test_sessions

print("\n" + "=" * 65)
print(f"Số session phân bổ - Train: {len(train_sessions)}, Val: {len(val_sessions)}, Test: {len(test_sessions)}")
print(f"Giao giữa Train & Val: {len(overlap_train_val)} (kỳ vọng 0)")
print(f"Giao giữa Train & Test: {len(overlap_train_test)} (kỳ vọng 0)")
print(f"Giao giữa Val & Test: {len(overlap_val_test)} (kỳ vọng 0)")
assert len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0, "Phát hiện rò rỉ phiên quay!"
print(">> XÁC NHẬN: Không có session nào xuất hiện ở hai split khác nhau! Dữ liệu phân chia hoàn hảo.")
print("=" * 65)

In [ ]:
# 4. Trực quan hóa 6 ảnh mẫu kèm nhãn bounding box gốc
CLASS_COLORS = {
    "pothole": "lime",       # Ổ gà - xanh nõn chuối
    "crack": "red",          # Vết nứt - đỏ
    "manhole": "cyan"        # Nắp cống - xanh da trời
}

# Chọn 6 ảnh mẫu có đối tượng từ tập train
samples_to_show = [r for r in splits_data["train"] if len(r["objects"]) >= 2][:6]
if len(samples_to_show) < 6:
    samples_to_show = [r for r in splits_data["train"] if len(r["objects"]) > 0][:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, sample in enumerate(samples_to_show):
    img_path = Path("data") / sample["image_path"]
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    draw = ImageDraw.Draw(img)
    
    for obj in sample["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
        col = CLASS_COLORS.get(obj["label"], "yellow")
        draw.rectangle([x1, y1, x2, y2], outline=col, width=3)
        draw.text((x1 + 2, max(0, y1 - 12)), obj["label"], fill=col)
        
    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"ID: {sample['image_id']}\nSession: {sample['session']}\nBoxes: {len(sample['objects'])}", fontsize=9)
    ax.axis("off")

plt.suptitle("6 ảnh mẫu từ tập Train kèm Bounding Box gốc", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

def yolo_to_box(label, yolo):
    """Chuyển đổi nhãn YOLO [xc, yc, w, h] chuẩn hoá [0,1] sang JSON bbox_2d thang 0-1000."""
    xc, yc, w, h = yolo
    x1 = round((xc - w / 2) * 1000)
    y1 = round((yc - h / 2) * 1000)
    x2 = round((xc + w / 2) * 1000)
    y2 = round((yc + h / 2) * 1000)
    # Kẹp giá trị vào khoảng [0, 1000]
    x1 = max(0, min(1000, x1))
    y1 = max(0, min(1000, y1))
    x2 = max(0, min(1000, x2))
    y2 = max(0, min(1000, y2))
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """Chuyển đổi danh sách đối tượng sang JSON format chuẩn: [{"label": ..., "bbox_2d": [...]}]
    Sắp xếp các box theo thứ tự (y1, x1) để thứ tự sinh token ổn định từ trên xuống dưới, từ trái sang phải.
    """
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = (
    "Locate all road damage objects in the image. "
    "Classes: pothole, crack, manhole. "
    "Output format: JSON list of objects [{\"label\": \"...\", \"bbox_2d\": [x1, y1, x2, y2]}]. "
    "Coordinates x1, y1, x2, y2 must be integers on a 0-1000 scale relative to image dimensions. "
    "If no damage is detected, return []."
)

# Kiểm tra round-trip nghịch đảo
sample = splits_data["train"][0]
ans_str = objects_to_answer(sample["objects"])
print("PROMPT chuẩn:")
print(PROMPT)
print("\nVí dụ output JSON:", ans_str)

parsed = json.loads(ans_str)
print(f"Số box: {len(parsed)} == {len(sample['objects'])}")
for orig, recon in zip(sample["objects"], parsed):
    oxc, oyc, ow, oh = orig["yolo"]
    rx1, ry1, rx2, ry2 = recon["bbox_2d"]
    r_xc = (rx1 + rx2) / 2000.0
    r_yc = (ry1 + ry2) / 2000.0
    assert abs(oxc - r_xc) < 0.005 and abs(oyc - r_yc) < 0.005, "Lỗi toạ độ nghịch đảo!"
print(">> Round-trip test thành công: toạ độ 0-1000 khớp chính xác với YOLO gốc!")

## 3. E0: zero-shot

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoProcessor, Qwen3VLForConditionalGeneration
from scripts.labcommon import read_jsonl, write_jsonl
from tqdm import tqdm

# Cấu hình lượng tử hóa NF4 4-bit, giữ visual và lm_head ở float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

model_id = "Qwen/Qwen3-VL-4B-Instruct"
print(f"Đang tải {model_id}...")

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
model.eval()
print("Model E0 đã nạp thành công!")

# Suy luận zero-shot trên tập test (311 ảnh)
test_samples = read_jsonl("data/splits/test.jsonl")
Path("work").mkdir(parents=True, exist_ok=True)
batch_size = 4
e0_results = []

for i in tqdm(range(0, len(test_samples), batch_size), desc="Inference E0"):
    batch_items = test_samples[i : i + batch_size]
    batch_messages = []
    for item in batch_items:
        img_path = Path("data") / item["image_path"]
        img = Image.open(img_path).convert("RGB")
        msg = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": PROMPT}
            ]
        }]
        batch_messages.append(msg)
    
    inputs = processor.apply_chat_template(
        batch_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        padding=True
    ).to(model.device)
    
    prompt_len = inputs.input_ids.shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    
    gen_tokens = out[:, prompt_len:]
    raw_texts = processor.tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
    for item, raw in zip(batch_items, raw_texts):
        e0_results.append({"image_id": item["image_id"], "raw": raw.strip()})

write_jsonl("work/e0.jsonl", e0_results)
print(f"Đã lưu kết quả suy luận vào work/e0.jsonl ({len(e0_results)} mẫu)")

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
del model, inputs, out
import gc
gc.collect()
torch.cuda.empty_cache()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, prompt, processor):
        self.samples = read_jsonl(jsonl_path)
        self.prompt = prompt
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        img_path = Path("data") / item["image_path"]
        img = Image.open(img_path).convert("RGB")
        answer = objects_to_answer(item["objects"])
        
        user_msg = [{"type": "image", "image": img}, {"type": "text", "text": self.prompt}]
        assistant_msg = [{"type": "text", "text": answer}]
        
        prompt_conv = [{"role": "user", "content": user_msg}]
        prompt_enc = self.processor.apply_chat_template(
            prompt_conv,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        )
        
        full_conv = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg}
        ]
        full_enc = self.processor.apply_chat_template(
            full_conv,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt"
        )
        
        prompt_len = prompt_enc.input_ids.shape[1]
        assert torch.equal(full_enc.input_ids[:, :prompt_len], prompt_enc.input_ids), "Prompt không phải tiền tố của chuỗi đầy đủ!"
        
        labels = full_enc.input_ids.clone()
        labels[:, :prompt_len] = -100  # Chỉ tính loss trên phần câu trả lời của assistant
        
        item_dict = {k: v.squeeze(0) if k in ("input_ids", "attention_mask", "mm_token_type_ids") else v for k, v in full_enc.items()}
        item_dict["labels"] = labels.squeeze(0)
        return item_dict

def road_collate_fn(batch):
    pad_token_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id
    padded_input_ids = torch.nn.utils.rnn.pad_sequence([item["input_ids"] for item in batch], batch_first=True, padding_value=pad_token_id)
    padded_labels = torch.nn.utils.rnn.pad_sequence([item["labels"] for item in batch], batch_first=True, padding_value=-100)
    attention_mask = (padded_input_ids != pad_token_id).long()
    
    out_batch = {
        "input_ids": padded_input_ids,
        "attention_mask": attention_mask,
        "labels": padded_labels,
    }
    
    if "mm_token_type_ids" in batch[0]:
        out_batch["mm_token_type_ids"] = torch.nn.utils.rnn.pad_sequence([item["mm_token_type_ids"] for item in batch], batch_first=True, padding_value=0)
    if "pixel_values" in batch[0]:
        out_batch["pixel_values"] = torch.cat([item["pixel_values"] for item in batch], dim=0)
    if "image_grid_thw" in batch[0]:
        out_batch["image_grid_thw"] = torch.cat([item["image_grid_thw"] for item in batch], dim=0)
        
    return out_batch

train_dataset = RoadDamageDataset("data/splits/train.jsonl", PROMPT, processor)
sample_item = train_dataset[0]
ans_tokens = sample_item["labels"][sample_item["labels"] != -100]
print("Số lượng token answer được tính loss:", len(ans_tokens))
print("Nội dung giải mã tokens answer:", processor.tokenizer.decode(ans_tokens, skip_special_tokens=False))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name="E1", rank=16):
    """Thiết lập LoRA và đóng băng/mở projector theo GUIDE bước 5."""
    for param in model.parameters():
        param.requires_grad = False
    
    model.enable_input_require_grads()
    
    # Target modules regex
    llm_pattern = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$"
    vit_pattern = r".*visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2))$"
    
    if setup_name in ("E1", "E2"):
        target_pattern = llm_pattern
    elif setup_name == "E3":
        target_pattern = f"({llm_pattern})|({vit_pattern})"
    else:
        raise ValueError(f"Setup không hợp lệ: {setup_name}")
        
    lora_config = LoraConfig(
        r=rank,
        lora_alpha=2 * rank,
        lora_dropout=0.05,
        target_modules=target_pattern,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_model = get_peft_model(model, lora_config)
    
    # Mở đầy đủ projector ở float32 cho E2 và E3
    if setup_name in ("E2", "E3"):
        proj_params = 0
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                proj_params += param.numel()
        print(f"[{setup_name}] Đã mở Projector ({proj_params:,} tham số ở float32)")
        
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup_name}] Trainable params: {trainable_params:,} / {all_params:,} ({trainable_params / all_params * 100:.2f}%)")
    return peft_model, trainable_params

## 6. Vòng huấn luyện

In [ ]:
import time
from torch.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup

def train_setup(setup_name="E1", max_steps=300, batch_size=2, grad_accum=4, lr=2e-4, rank=16, run_name=None):
    run_name = run_name or setup_name
    print(f"\n================ BẮT ĐẦU HUẤN LUYỆN {setup_name} ================")
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    peft_model, num_trainable = apply_setup(base_model, setup_name, rank)
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_model.config.use_cache = False
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=road_collate_fn
    )
    
    trainable_p = [p for p in peft_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_p, lr=lr, weight_decay=0.0)
    warmup_steps = int(max_steps * 0.05)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max_steps)
    scaler = GradScaler("cuda")
    
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    
    step = 0
    accum_step = 0
    optimizer.zero_grad()
    data_iter = iter(train_loader)
    peft_model.train()
    pbar = tqdm(total=max_steps, desc=f"Train {setup_name}")
    
    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)
            
        batch = {k: v.to(peft_model.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        with autocast("cuda", dtype=torch.float16):
            loss = peft_model(**batch).loss / grad_accum
            
        assert torch.isfinite(loss), f"Tràn số loss nan/inf tại step {step}!"
        scaler.scale(loss).backward()
        accum_step += 1
        
        if accum_step % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_p, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item() * grad_accum:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            
            if step % 100 == 0 or step == max_steps:
                drive_base = Path("/content/drive/MyDrive")
                save_dir = (drive_base / f"checkpoints/{run_name}") if drive_base.exists() else Path(f"work/checkpoints/{run_name}")
                save_dir.mkdir(parents=True, exist_ok=True)
                peft_model.save_pretrained(str(save_dir / f"adapter_step_{step}"))
                if setup_name in ("E2", "E3"):
                    proj_state = {n: p.data.cpu() for n, p in peft_model.named_parameters()
                                  if "visual.merger" in n or "visual.deepstack_merger_list" in n}
                    torch.save(proj_state, save_dir / f"projector_step_{step}.pt")
                    
    pbar.close()
    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    
    run_meta = {
        "setup": setup_name,
        "trainable_params": num_trainable,
        "peak_vram_gb": round(peak_vram_gb, 2),
        "train_minutes": round(train_minutes, 2)
    }
    Path(f"work/{run_name.lower()}_run.json").write_text(json.dumps(run_meta, indent=2))
    if drive_base.exists():
        (drive_base / f"checkpoints/{run_name}/run.json").write_text(json.dumps(run_meta, indent=2))
    print(f"[{setup_name}] Hoàn thành: {train_minutes:.2f} phút | Peak VRAM: {peak_vram_gb:.2f} GB")
    del optimizer, scheduler, scaler, peft_model, base_model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return run_meta

## 7. Suy luận và chấm E1–E3

In [ ]:
from peft import PeftModel

def run_eval_setup(setup_name="E1", run_meta=None, run_name=None):
    run_name = run_name or setup_name
    print(f"\n================ SUY LUẬN {setup_name} ================")
    if run_meta is None:
        meta_file = Path(f"work/{run_name.lower()}_run.json")
        if not meta_file.exists():
            meta_file = Path(f"/content/drive/MyDrive/checkpoints/{run_name}/run.json")
        if not meta_file.exists():
            raise FileNotFoundError(f"Thiếu metadata huấn luyện: {meta_file}")
        run_meta = json.loads(meta_file.read_text())
            
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    drive_base = Path("/content/drive/MyDrive")
    if (drive_base / f"checkpoints/{run_name}/adapter_step_300").exists():
        adapter_path = str(drive_base / f"checkpoints/{run_name}/adapter_step_300")
        proj_path = drive_base / f"checkpoints/{run_name}/projector_step_300.pt"
    else:
        adapter_path = f"work/checkpoints/{run_name}/adapter_step_300"
        proj_path = Path(f"work/checkpoints/{run_name}/projector_step_300.pt")
    if setup_name in ("E2", "E3") and not proj_path.exists():
        raise FileNotFoundError(f"Thiếu projector đã huấn luyện: {proj_path}")
        
    eval_model = PeftModel.from_pretrained(base_model, adapter_path)
    
    if setup_name in ("E2", "E3"):
        proj_state = torch.load(proj_path, map_location="cpu", weights_only=True)
        projector = {n: p for n, p in eval_model.named_parameters() if "visual.merger" in n or "visual.deepstack_merger_list" in n}
        assert proj_state.keys() == projector.keys(), "Checkpoint projector không khớp model"
        for n, p in projector.items():
            p.data.copy_(proj_state[n].to(p.device, p.dtype))
        print(f"[{setup_name}] Đã nạp lại projector weights!")
            
    eval_model.eval()
    test_samples = read_jsonl("data/splits/test.jsonl")
    pred_results = []
    
    for i in tqdm(range(0, len(test_samples), 4), desc=f"Eval {setup_name}"):
        batch_items = test_samples[i : i + 4]
        batch_messages = []
        for item in batch_items:
            img_path = Path("data") / item["image_path"]
            img = Image.open(img_path).convert("RGB")
            msg = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
            batch_messages.append(msg)
            
        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True
        ).to(eval_model.device)
        
        prompt_len = inputs.input_ids.shape[1]
        with torch.no_grad():
            out = eval_model.generate(**inputs, max_new_tokens=256, do_sample=False)
            
        gen_tokens = out[:, prompt_len:]
        raw_texts = processor.tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        for item, raw in zip(batch_items, raw_texts):
            pred_results.append({"image_id": item["image_id"], "raw": raw.strip()})
            
    pred_path = f"work/{run_name.lower()}.jsonl"
    write_jsonl(pred_path, pred_results)
    
    out_evidence = f"report/evidence/{run_name.lower()}.json"
    cmd = (
        f"python scripts/score.py --pred {pred_path} --name {run_name} --out {out_evidence} "
        f"--meta trainable_params={run_meta['trainable_params']} "
        f"--meta peak_vram_gb={run_meta['peak_vram_gb']} "
        f"--meta train_minutes={run_meta['train_minutes']}"
    )
    print("Lệnh chấm điểm:", cmd)
    import subprocess
    subprocess.run(cmd.split(), check=True)
    del eval_model, base_model, inputs, out
    import gc
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Chạy lần lượt các thí nghiệm (bỏ comment để chạy trên Colab T4):
# meta_e1 = train_setup("E1")
# run_eval_setup("E1", meta_e1)

# meta_e2 = train_setup("E2")
# run_eval_setup("E2", meta_e2)

# meta_e3 = train_setup("E3")
# run_eval_setup("E3", meta_e3)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: chọn setup tốt nhất E1-E3 theo F1 test, chỉ đổi rank.
# best_setup = max(("E1", "E2", "E3"), key=lambda s: json.loads(Path(f"report/evidence/{s.lower()}.json").read_text())["micro"]["f1"])
# for rank in (4, 64):
#     name = f"e4_rank_{rank}"
#     meta = train_setup(best_setup, rank=rank, run_name=name)
#     run_eval_setup(best_setup, meta, run_name=name)

# E5 (Bonus +10): Định nghĩa hàm Reward IoU cho GRPO
from scripts.score import parse_output, iou_xyxy

def compute_reward(completion, gt_boxes):
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
    
    # Match greedy theo IoU >= 0.5 cùng nhãn
    matched_iou_sum = 0.0
    used_gt = set()
    used_pred = set()
    
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g["label"] == p["label"]:
                iou = iou_xyxy(g["bbox"], p["bbox"])
                if iou >= 0.5:
                    pairs.append((iou, i, j))
    pairs.sort(reverse=True, key=lambda x: x[0])
    
    for iou, i, j in pairs:
        if i not in used_gt and j not in used_pred:
            used_gt.add(i)
            used_pred.add(j)
            matched_iou_sum += iou
            
    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * (matched_iou_sum / denom)

# Kiểm tra hàm reward
dummy_gt = [{"label": "pothole", "bbox": (100, 100, 200, 200)}]
dummy_pred_perfect = '[{"label": "pothole", "bbox_2d": [100, 100, 200, 200]}]'
dummy_pred_corrupt = 'invalid json'
print("Reward hoàn hảo:", compute_reward(dummy_pred_perfect, dummy_gt))
print("Reward JSON hỏng:", compute_reward(dummy_pred_corrupt, dummy_gt))

In [ ]:
# Kiểm tra toàn diện các điều kiện nghiệm thu trước khi nộp bài
!python scripts/check_submission.py